# Reproduce the paper's trial analysis from a raw-only NWB file

This notebook starts from an NWB file that holds only the raw photodetector signal, as written by `hadjas-2025 convert <data folder> <output folder> --processed none`, and:

1. computes the processed signals from the raw signal (demodulation, resampling to 100 Hz and dF/F, `hadjas_2025.analysis.processing`) and writes them to the file, as the conversion does with `processed: recomputed`;
2. runs the trial analysis of the lab's MATLAB scripts on the dF/F (`hadjas_2025.analysis.trial_analysis`) and writes its results to the file;
3. reads the results back from the file to plot the trials and the mean responses and to tabulate the window statistics.

`validate_processing.ipynb` shows how the recomputed signals compare with the lab's. Given the lab's dF/F, the trial analysis gives the lab's trials and statistics to floating point precision.

## 1. Add the processed signals and the trial analysis to the file

The results are added to a copy, so that the raw-only file stays as it is. The cohort of the subject (a key of `cohorts` in `metadata/session_config.yaml`) gives the names and metadata of the processed series.

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import TwoSlopeNorm
from pynwb import NWBHDF5IO

from hnasko_lab_to_nwb.hadjas_2025.analysis.processed_series import add_recomputed_processed_series
from hnasko_lab_to_nwb.hadjas_2025.analysis.trial_analysis import add_trial_analysis
from hnasko_lab_to_nwb.hadjas_2025.convert_session import get_led_modulation, load_cohort_metadata

raw_nwbfile_path = Path("nwb_files/sub-C4561_ses-varying_frequencies.nwb")
recording_type = "SN pan GABA recordings"

nwbfile_path = raw_nwbfile_path.with_name(raw_nwbfile_path.stem + "_analysed.nwb")
shutil.copy(raw_nwbfile_path, nwbfile_path)

The raw series has the metadata key `Fi1r_<recording site>`, and the dF/F series the key `dF_<site>`. The trial analysis aligns the dF/F on the trains of the optogenetic epochs table, groups the trials by train duration and pulse frequency, and adds two tables to the `trial_analysis` processing module.

In [ ]:
metadata = load_cohort_metadata(recording_type)
with NWBHDF5IO(nwbfile_path, "a") as io:
    nwbfile = io.read()
    raw_metadata_key = next(
        key
        for key, series_metadata in metadata["FiberPhotometry"].items()
        if key.startswith("Fi1r_") and series_metadata["name"] in nwbfile.acquisition
    )
    site = raw_metadata_key.removeprefix("Fi1r_")
    add_recomputed_processed_series(
        nwbfile=nwbfile,
        metadata=metadata,
        raw_metadata_key=raw_metadata_key,
        led_modulation=get_led_modulation(nwbfile.session_start_time),
    )
    add_trial_analysis(nwbfile=nwbfile, dff_series_name=metadata["FiberPhotometry"][f"dF_{site}"]["name"], site=site)
    io.write(nwbfile)

## 2. Read the results back

`trial_aligned_dff_<site>` has one row per train, with its dF/F and normalized dF/F from 5 s before to 10 s after its onset at 100 Hz. The normalized dF/F is the dF/F minus its mean over the second before the onset. `trial_window_statistics_<site>` has one row per train duration, pulse frequency and time window.

In [ ]:
with NWBHDF5IO(nwbfile_path, "r") as io:
    nwbfile = io.read()
    subject_id, session_id = nwbfile.subject.subject_id, nwbfile.session_id
    module = nwbfile.processing["trial_analysis"]
    trials = module[f"trial_aligned_dff_{site}"].to_dataframe()
    statistics = module[f"trial_window_statistics_{site}"].to_dataframe()

time = np.arange(-500, 1001) / 100.0  # seconds from the onset
durations = list(dict.fromkeys(trials["train_duration"]))
frequencies = sorted(trials["stimulus_frequency"].unique())
trials.head()

## 3. Trials

Normalized dF/F of every trial, one panel per train duration and pulse frequency. The dashed line is the train onset and the dotted line its end.

In [ ]:
duration_in_seconds = {"250ms": 0.25, "1s": 1.0, "4s": 4.0}
values = np.stack(trials["normalized_dff"])
limit = np.nanpercentile(np.abs(values), 99)
figure, axes = plt.subplots(
    len(durations),
    len(frequencies),
    figsize=(3.2 * len(frequencies), 2.4 * len(durations)),
    sharex=True,
    squeeze=False,
    layout="constrained",
)
for row, duration in enumerate(durations):
    for column, frequency in enumerate(frequencies):
        ax = axes[row, column]
        condition = trials[(trials["train_duration"] == duration) & (trials["stimulus_frequency"] == frequency)]
        if len(condition) == 0:
            ax.set_axis_off()
            continue
        image = ax.imshow(
            np.stack(condition["normalized_dff"]),
            aspect="auto",
            cmap="RdBu_r",
            norm=TwoSlopeNorm(vcenter=0, vmin=-limit, vmax=limit),
            extent=[time[0], time[-1], len(condition) + 0.5, 0.5],
            interpolation="none",
        )
        ax.axvline(0, color="0.2", ls="--", lw=1)
        ax.axvline(duration_in_seconds.get(duration, 0), color="0.2", ls=":", lw=1)
        ax.set_title(f"{duration}, {frequency:g} Hz", fontsize=9)
        ax.set_xlabel("time from onset (s)" if row == len(durations) - 1 else "")
        ax.set_ylabel("trial" if column == 0 else "")
figure.colorbar(image, ax=axes, shrink=0.6, label="normalized dF/F (%)")
figure.suptitle(f"{subject_id}, {session_id}");

## 4. Mean responses

Normalized dF/F averaged over the trials of each condition, with the standard error of the mean, one panel per train duration and one line per pulse frequency (darker for higher frequencies).

In [ ]:
colors = plt.cm.Blues(np.linspace(0.45, 0.95, len(frequencies)))
figure, axes = plt.subplots(
    1, len(durations), figsize=(4.5 * len(durations), 3.4), sharey=True, squeeze=False, layout="constrained"
)
for ax, duration in zip(axes[0], durations):
    for color, frequency in zip(colors, frequencies):
        condition = trials[(trials["train_duration"] == duration) & (trials["stimulus_frequency"] == frequency)]
        if len(condition) == 0:
            continue
        condition_values = np.stack(condition["normalized_dff"])
        mean = condition_values.mean(axis=0)
        sem = condition_values.std(axis=0, ddof=1) / np.sqrt(len(condition_values))
        ax.fill_between(time, mean - sem, mean + sem, color=color, alpha=0.2, lw=0)
        ax.plot(time, mean, color=color, lw=2, label=f"{frequency:g} Hz")
    ax.axvspan(0, duration_in_seconds.get(duration, 0), color="0.9", zorder=0)
    ax.axhline(0, color="0.6", lw=0.8)
    ax.set(title=f"{duration} trains", xlabel="time from onset (s)", xlim=(-2, 8))
axes[0, 0].set_ylabel("normalized dF/F (%)")
figure.legend(*axes[0, 0].get_legend_handles_labels(), title="pulse frequency", fontsize=8, loc="outside right center")
figure.suptitle(f"{subject_id}, {session_id}");

## 5. Window statistics

The statistics of the lab's scripts in each time window (named as in the scripts; a window `stmA_B` holds the samples after A and up to B from the onset). For example, the mean normalized dF/F during the train, per train duration and pulse frequency:

In [ ]:
window_during_train = {"250ms": "stm0_250ms", "1s": "stm0_1s", "4s": "stm0_4s"}
during_train = statistics[statistics["window"] == statistics["train_duration"].map(window_during_train)]
during_train.pivot(index="stimulus_frequency", columns="train_duration", values="mean")[durations]

All statistics of all windows, for example to export to a spreadsheet with `statistics.to_csv(...)`:

In [ ]:
statistics